# Data Preparation — Smart Energy Optimizer
UCI Individual Household Electric Power Consumption dataset.
Fixed version: uses Global_active_power as target (not derived from sub-metering).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print('Libraries loaded.')

## Step 1 — Load dataset

In [ ]:
df = pd.read_csv(
    'household_power_consumption.txt',
    sep=';',
    parse_dates={'Datetime': ['Date', 'Time']},
    infer_datetime_format=True,
    na_values='?',
    low_memory=False
)

print('Shape:', df.shape)
df.head()

## Step 2 — Handle missing values (interpolate, not drop)

In [ ]:
numeric_cols = [
    'Global_active_power', 'Global_reactive_power', 'Voltage',
    'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3'
]

# Cast to float first
df[numeric_cols] = df[numeric_cols].astype(float)

print('Missing before interpolation:')
print(df[numeric_cols].isna().sum())

# Linear interpolation — fills gaps using neighbouring readings
df[numeric_cols] = df[numeric_cols].interpolate(method='linear', limit_direction='both')

# Drop any rows still missing after interpolation (very rare)
df.dropna(inplace=True)

print('\nMissing after interpolation:')
print(df[numeric_cols].isna().sum())
print('\nRows remaining:', len(df))

## Step 3 — Group by day and compute the TARGET correctly

**Key fix:** `Global_active_power` is in kW measured every minute.
To convert to kWh: sum all minute readings, then divide by 60.
This is the TRUE total energy consumed — not derived from sub-metering.

In [ ]:
df['Date'] = df['Datetime'].dt.date

daily = df.groupby('Date').agg(
    Total_kWh      = ('Global_active_power', lambda x: x.sum() / 60),  # kW * 1min / 60 = kWh
    Sub_metering_1 = ('Sub_metering_1', 'sum'),  # Wh
    Sub_metering_2 = ('Sub_metering_2', 'sum'),  # Wh
    Sub_metering_3 = ('Sub_metering_3', 'sum'),  # Wh
).reset_index()

# Convert sub-metering from Wh to kWh
daily['Kitchen_kWh']     = daily['Sub_metering_1'] / 1000
daily['Laundry_kWh']     = daily['Sub_metering_2'] / 1000
daily['AC_Geyser_kWh']   = daily['Sub_metering_3'] / 1000

print('Daily data shape:', daily.shape)
print('Sample Total_kWh range:', daily['Total_kWh'].min().round(2), '—', daily['Total_kWh'].max().round(2))
daily.head()

## Step 4 — Feature engineering

In [ ]:
daily['Date']      = pd.to_datetime(daily['Date'])
daily['Month']     = daily['Date'].dt.month
daily['DayOfWeek'] = daily['Date'].dt.dayofweek

# Pakistan summer: May–September (month 5–9)
daily['Season'] = daily['Month'].apply(lambda m: 1 if m in [5, 6, 7, 8, 9] else 0)

# Appliance-level features derived from sub-metering
# Split AC_Geyser (sub_metering_3) — 70% AC, 30% Geyser (best estimate without smart meters)
daily['AC_kWh']    = daily['AC_Geyser_kWh'] * 0.70
daily['Geyser_kWh']= daily['AC_Geyser_kWh'] * 0.30
daily['Fridge_kWh']= daily['Laundry_kWh']   * 0.60
daily['WM_kWh']    = daily['Laundry_kWh']   * 0.40

# Estimated hours (wattage constants from dataset household — French, 220V)
daily['AC_Hours']             = (daily['AC_kWh']     / 1.8).clip(0, 24).round(1)
daily['Geyser_Hours']         = (daily['Geyser_kWh'] / 2.5).clip(0, 8).round(1)
daily['WashingMachine_Hours'] = (daily['WM_kWh']     / 0.8).clip(0, 6).round(1)
daily['Fridge_Hours']         = 24  # always on

# AC tonnage estimate from daily consumption
def ac_tonnage(kwh):
    if kwh < 4:  return 1.0
    elif kwh < 8: return 1.5
    else:         return 2.0

daily['AC_Tonnage'] = daily['AC_kWh'].apply(ac_tonnage)

# Fridge size
daily['Fridge_Size'] = daily['Fridge_kWh'].apply(
    lambda x: 1 if x < 1 else (2 if x < 2 else 3)
)

# Family members estimate from total consumption
daily['Family_Members'] = daily['Total_kWh'].apply(
    lambda x: 2 if x < 8 else (4 if x < 15 else (6 if x < 25 else 8))
)

# Peak usage flag
daily['Peak_Usage'] = (daily['AC_kWh'] > 5).astype(int)

daily.head()

## Step 5 — Check target distribution

In [ ]:
print('Target (Total_kWh) stats:')
print(daily['Total_kWh'].describe().round(3))

daily['Total_kWh'].hist(bins=50, figsize=(10, 4), color='steelblue', edgecolor='white')
plt.title('Distribution of daily kWh consumption')
plt.xlabel('kWh per day')
plt.ylabel('Days')
plt.tight_layout()
plt.savefig('target_distribution.png', dpi=150)
plt.show()

## Step 6 — Select final ML columns and save

In [ ]:
ml_cols = [
    'AC_Hours', 'AC_Tonnage',
    'Fridge_Hours', 'Fridge_Size',
    'Geyser_Hours',
    'WashingMachine_Hours',
    'Family_Members',
    'Season', 'Month', 'DayOfWeek',
    'Peak_Usage',
    'Total_kWh',  # TARGET — real consumption, not derived from features
]

ml_data = daily[ml_cols].dropna()
print('Final ML dataset shape:', ml_data.shape)
print(ml_data.describe().round(3))

# Save with .csv extension (important!)
ml_data.to_csv('daily_appliance_usage.csv', index=False)
print('\nSaved as daily_appliance_usage.csv')